# Training and Prediction

### LinearSVC with TfidfVectorizer

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
import joblib
import os

# Load the merged data
df = pd.read_csv("cefr_final_merged.csv")

labels = ['A1', 'A2', 'B1', 'B2', 'C1']
df['label'] = df['CEFR'].map({l: i for i, l in enumerate(labels)})

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])


vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), lowercase=False)
X_train = vectorizer.fit_transform(train_df['text'])
X_test  = vectorizer.transform(test_df['text'])
y_train = train_df['label']
y_test  = test_df['label']

model = LinearSVC(C=1.0, class_weight='balanced', max_iter=1000, random_state=42)
model.fit(X_train, y_train)

print(f"ACCURACY: {accuracy_score(y_test, model.predict(X_test)):.4f}")

# Save model
os.makedirs("cefr_german_model", exist_ok=True)
joblib.dump(model,      "cefr_german_model/model.pkl")
joblib.dump(vectorizer, "cefr_german_model/vectorizer.pkl")
joblib.dump(labels,     "cefr_german_model/labels.pkl")
print("Model saved!")

ACCURACY: 0.8044
Model saved!


### Prediction

In [14]:
import joblib

# Load model
vectorizer = joblib.load("cefr_german_model/vectorizer.pkl")
model      = joblib.load("cefr_german_model/model.pkl")
labels     = ['A1', 'A2', 'B1', 'B2', 'C1']

def cefr(sentence):
    return labels[model.predict(vectorizer.transform([sentence]))[0]]

# prediction
sentence = input("Enter a German sentence: ").strip()

if not sentence:
    print("No input — goodbye!")
else:
    level = cefr(sentence)
    print(f"\nCEFR Level: {level}")

Enter a German sentence:  Es besteht kaum Zweifel daran, dass die zunehmende Digitalisierung sämtliche Lebensbereiche nachhaltig prägt.



CEFR Level: C1
